## Construcción variable objetivo y features

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Definir rutas
processed_path = "/workspaces/tumipay_analysis/data/processed"

# Cargar datasets limpios
clientes = pd.read_csv(f"{processed_path}/clientes_clean.csv")
creditos = pd.read_csv(f"{processed_path}/creditos_clean.csv")
pagos = pd.read_csv(f"{processed_path}/pagos_clean.csv")
eventos = pd.read_csv(f"{processed_path}/eventos_clean.csv")

In [5]:
# Construcción de variable objetivo

# Etiqueta de riesgo: mora > 30 días
riesgo_credito = pagos.groupby("credito_id")["dias_mora"].max().reset_index()
riesgo_credito["riesgo_mora_30"] = (riesgo_credito["dias_mora"] > 30).astype(int)

# Unir con créditos
creditos_feat = creditos.merge(riesgo_credito[["credito_id", "riesgo_mora_30"]],
                               on="credito_id", how="left")

In [6]:
# Feature engineering (usando solo variables disponibles en el momento de originación)

# Variables de clientes
clientes_feat = clientes[[
    "cliente_id", "edad", "genero", "estrato", "nivel_educativo",
    "ocupacion", "ingreso_mensual_estimado", "score_externo",
    "tiene_producto_ahorro", "numero_dependientes", "dispositivo_principal"
]]

# Variables de créditos
creditos_feat = creditos_feat[[
    "credito_id", "cliente_id", "fecha_desembolso", "producto_credito",
    "monto_credito", "plazo_meses", "tasa_interes_mensual",
    "valor_cuota_pactada", "canal_originacion", "score_interno_originacion",
    "relacion_cuota_ingreso", "politica_aprobacion", "estado_credito_operativo",
    "riesgo_mora_30"
]]

# Variables de eventos
eventos_pre = eventos.merge(creditos[["credito_id","cliente_id","fecha_desembolso"]],
                            on="cliente_id", how="left")
eventos_pre = eventos_pre[eventos_pre["fecha_evento"] <= eventos_pre["fecha_desembolso"]]

eventos_agg = eventos_pre.groupby("cliente_id").agg({
    "evento_id": "count",
    "duracion_sesion_seg": "mean"
}).reset_index()

eventos_agg.rename(columns={
    "evento_id": "total_eventos_pre",
    "duracion_sesion_seg": "duracion_promedio_pre"
}, inplace=True)

In [ ]:
# Merge general

df_model = creditos_feat.merge(clientes_feat, on="cliente_id", how="left")
df_model = df_model.merge(eventos_agg, on="cliente_id", how="left")

In [ ]:
# Feature selection

# Variables numéricas y categóricas separadas
numeric_cols = ["edad", "ingreso_mensual_estimado", "score_externo",
                "monto_credito", "plazo_meses", "tasa_interes_mensual",
                "valor_cuota_pactada", "score_interno_originacion",
                "relacion_cuota_ingreso", "numero_dependientes",
                "total_eventos_pre", "duracion_promedio_pre"]

categorical_cols = ["genero", "estrato", "nivel_educativo",
                    "ocupacion", "producto_credito", "canal_originacion",
                    "politica_aprobacion", "estado_credito_operativo",
                    "dispositivo_principal"]

# Documentar: estas columnas se usarán en el modelado con encoding y normalización

In [9]:
# Guardar dataset final

df_model.to_csv(f"{processed_path}/model_input.csv", index=False)

print("Dataset final para modelado guardado en /data/processed/model_input.csv")

Dataset final para modelado guardado en /data/processed/model_input.csv
